# Deploy the VSS Orchestrator MCP and Deploy VSS from OpenClaw Within NemoClaw

Run this notebook on the same host that contains the VSS repo, can run the VSS Orchestrator MCP server, and can reach the OpenClaw/NemoClaw sandbox.

This notebook walks through nine steps:

1. Checking the required prerequisites on the host.
2. Installing and configuring the NGC CLI for local NIM-backed deployment.
3. Authenticating Docker to `nvcr.io`.
4. Preparing the `services/agent/` Python environment.
5. Reviewing the orchestrator MCP configuration.
6. Referring to [`deploy_nemoclaw_vss.ipynb`](./deploy_nemoclaw_vss.ipynb) for NemoClaw/OpenClaw sandbox policy setup.
7. Starting the VSS Orchestrator MCP server.
8. Using OpenClaw or the manual helper cells to invoke the orchestrator tools and deploy VSS.
9. Stopping the VSS Orchestrator MCP server when you are done.

**Important:** run [`deploy_nemoclaw_vss.ipynb`](./deploy_nemoclaw_vss.ipynb) first if OpenClaw/NemoClaw is not already configured on this instance.


## 1. Pre-requisites and pre-steps

Before you start this notebook, make sure these are already in place on the host:

- The repo exists on this host, typically at `/home/ubuntu/video-search-and-summarization`.
- [`deploy_nemoclaw_vss.ipynb`](./deploy_nemoclaw_vss.ipynb) has already been run successfully, or you already have a working OpenClaw/NemoClaw sandbox.
- Docker is installed and working.
- `openshell` is installed and can access the sandbox created by NemoClaw.
- `uv` and Python 3 are available so the `services/agent/` dependencies can be installed.
- If you plan to deploy local NIM-backed VSS profiles through the orchestrator tools, export `NGC_CLI_API_KEY` before running Steps 2 and 3.
- If you plan to use remote NVIDIA endpoints in profile generation, export `NVIDIA_API_KEY` before running the orchestrator tools that need it.

The next two sections are the actual pre-steps for local NIM-backed deployment:

1. Install the NGC CLI if it is missing and configure it with `NGC_CLI_API_KEY`.
2. Run `docker login nvcr.io` with `NGC_CLI_API_KEY`.

Notes:

- The orchestrator MCP server runs on the host.
- OpenClaw reaches that host-side MCP server from inside the sandbox through `http://host.openshell.internal:<port>/mcp`.
- The default port used below is `9902`.


In [ ]:
NGC_CLI_API_KEY = ""     # NGC legacy key
NVIDIA_API_KEY = ""  # nvapi-xxxxx
PROFILE = "base"  # supported profiles: ["base", "search", "alerts", "lvs"]
HARDWARE_PROFILE = "RTXPRO6000BW"
EXTRA_ENV_OVERRIDES = []  # example: ["VLM_MODE=local_shared"]

In [ ]:
import os
import shutil
from pathlib import Path

VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", "/home/ubuntu/video-search-and-summarization")).resolve()
AGENT_DIR = VSS_REPO_DIR / "services" / "agent"
ORCHESTRATOR_DIR = AGENT_DIR / "src" / "vss_agents" / "orchestrator"
MCP_CONFIG_PATH = ORCHESTRATOR_DIR / "vss_orchestrator_mcp_config.yml"
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "demo")
MCP_HOST = os.environ.get("VSS_ORCHESTRATOR_MCP_HOST", "0.0.0.0")
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9902"))
MCP_URL = f"http://{MCP_HOST}:{MCP_PORT}/mcp"
SANDBOX_MCP_URL = f"http://host.openshell.internal:{MCP_PORT}/mcp"
os.environ["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY

print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("AGENT_DIR:", AGENT_DIR)
print("ORCHESTRATOR_DIR:", ORCHESTRATOR_DIR)
print("MCP_CONFIG_PATH:", MCP_CONFIG_PATH)
print("POLICY_PATH:", POLICY_PATH)
print("NEMOCLAW_SANDBOX_NAME:", NEMOCLAW_SANDBOX_NAME)
print("MCP_URL:", MCP_URL)
print("SANDBOX_MCP_URL:", SANDBOX_MCP_URL)
print()

checks = {
    "repo directory": VSS_REPO_DIR.is_dir(),
    "agent directory": AGENT_DIR.is_dir(),
    "orchestrator directory": ORCHESTRATOR_DIR.is_dir(),
    "MCP config": MCP_CONFIG_PATH.is_file(),
    "NemoClaw policy": POLICY_PATH.is_file(),
    "python3": shutil.which("python3") is not None,
    "uv": shutil.which("uv") is not None,
    "docker": shutil.which("docker") is not None,
    "ngc": shutil.which("ngc") is not None,
    "openshell": shutil.which("openshell") is not None,
    "curl": shutil.which("curl") is not None,
    "NVIDIA_API_KEY set": bool(NVIDIA_API_KEY.strip()),
    "NGC_CLI_API_KEY set": bool(NGC_CLI_API_KEY.strip()),
}

GREEN = "\033[32m"
RED = "\033[31m"
RESET = "\033[0m"

for label, ok in checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")

## 2. Install and configure NGC CLI

If you plan to deploy local NIM-backed VSS profiles through the orchestrator tools, the host needs the NGC CLI.

This pre-step checks whether `ngc` is already installed, installs it if needed, and writes the local NGC CLI config using `NGC_CLI_API_KEY`.


In [ ]:
import os
import platform
import shutil
import subprocess


def run(cmd: str) -> str:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stderr}\n{result.stdout}")
    return result.stdout.strip()


ngc_path = shutil.which("ngc")
if ngc_path:
    version = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {version}")
else:
    arch = platform.machine()
    filename = "ngccli_linux_arm64.zip" if arch in ("aarch64", "arm64") else "ngccli_linux.zip"
    ngc_cli_version = "4.13.0"
    url = (
        "https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/"
        f"versions/{ngc_cli_version}/files/{filename}"
    )

    print(f"Installing NGC CLI {ngc_cli_version}...")
    run(f"cd /tmp && wget -q --content-disposition '{url}' -O ngc_cli.zip")

    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(
            f"NGC CLI download failed: /tmp/ngc_cli.zip is only {size} bytes."
        )

    run("cd /tmp && unzip -o ngc_cli.zip")
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    version = run("ngc --version 2>&1 | head -1")
    print(f"Installed NGC CLI: {version}")


# Configure NGC CLI with API key and org
if not NGC_CLI_API_KEY:
    raise RuntimeError(
        "NGC_CLI_API_KEY is not set. Set it in the notebook settings cell or export it "
        "in the environment before running this step."
    )

print("Configuring NGC CLI...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)

with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = nvstaging
""")

print("NGC CLI configured.")
print(run("ngc config current"))


## 3. Docker login to `nvcr.io`

If you plan to deploy local NIM-backed VSS profiles, authenticate Docker to the NVIDIA Container Registry before using the orchestrator deployment tools.

This pre-step runs `docker login nvcr.io` with `NGC_CLI_API_KEY`.


In [ ]:
import os
import subprocess

ngc_cli_api_key = os.environ.get("NGC_CLI_API_KEY")
if not ngc_cli_api_key:
    raise RuntimeError("NGC_CLI_API_KEY is not set. Export it before running this cell.")

login_result = subprocess.run(
    [
        "docker",
        "login",
        "nvcr.io",
        "--username",
        "$oauthtoken",
        "--password",
        ngc_cli_api_key,
    ],
    capture_output=True,
    text=True,
)
if login_result.returncode != 0:
    raise RuntimeError(f"Docker login to nvcr.io failed\n{login_result.stderr}")

print("Docker login to nvcr.io: OK")


## 4. Prepare the `services/agent/` environment

The orchestrator MCP server is part of the VSS agent package, so install the Python environment in `services/agent/` first.

This runs `uv sync` from the agent directory.


In [ ]:
import os
import shutil
import subprocess

if shutil.which("uv") is None:
    raise RuntimeError("uv is not installed. Install uv first, then re-run this cell.")

uv_env = os.environ.copy()
uv_env.pop("VIRTUAL_ENV", None)
subprocess.run(["uv", "sync"], cwd=str(AGENT_DIR), env=uv_env, check=True)
venv_info = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        "import os, sys; print('uv VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))",
    ],
    cwd=str(AGENT_DIR),
    env=uv_env,
    check=True,
    capture_output=True,
    text=True,
)
print(venv_info.stdout.strip())
print("Environment is ready.")


## 5. Review the orchestrator MCP configuration

The orchestrator server reads its runtime configuration from [`services/agent/src/vss_agents/orchestrator/vss_orchestrator_mcp_config.yml`](../../services/agent/src/vss_agents/orchestrator/vss_orchestrator_mcp_config.yml).

At minimum, confirm these paths are correct for this host before starting the server:

- `mdx_data_dir`
- `output_dir`
- `deployments_dir`
- `source_compose_yaml`
- `source_env`

If this repo is still under `/home/ubuntu/video-search-and-summarization`, the defaults should already be correct.


In [ ]:
import re

config_text = MCP_CONFIG_PATH.read_text()
for key in ["mdx_data_dir", "output_dir", "deployments_dir", "source_compose_yaml", "source_env"]:
    match = re.search(rf'^\s*{re.escape(key)}:\s*"?(.*?)"?\s*$', config_text, re.MULTILINE)
    print(f"{key}: {match.group(1) if match else 'NOT FOUND'}")


## 6. Run [`deploy_nemoclaw_vss.ipynb`](./deploy_nemoclaw_vss.ipynb) first

<span style="color:red">Run [`deploy_nemoclaw_vss.ipynb`](./deploy_nemoclaw_vss.ipynb) before continuing. It sets the sandbox policy OpenClaw needs so the sandbox can reach the host MCP server at `http://host.openshell.internal:&lt;MCP_PORT&gt;/mcp` (default `9902`).</span>

## 7. Start the VSS Orchestrator MCP server

This cell stops any existing MCP server tracked in notebook memory, then starts a fresh server in the background and writes logs under `.orchestrator-artifacts/`.

After startup, the next cell prints the host-side endpoint from the current `MCP_HOST` and `MCP_PORT` settings. From inside the sandbox, use `http://host.openshell.internal:&lt;MCP_PORT&gt;/mcp`. The default port is `9902`.


In [ ]:
import os
import signal
import subprocess
import time

ARTIFACT_DIR = VSS_REPO_DIR / ".orchestrator-artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / "vss_orchestrator_mcp.log"

existing_pid = globals().get("VSS_ORCHESTRATOR_MCP_PID")
if existing_pid:
    try:
        os.kill(existing_pid, signal.SIGTERM)
        print(f"Stopped existing MCP server PID {existing_pid}")
        time.sleep(2)
    except ProcessLookupError:
        print(f"Recorded MCP server PID {existing_pid} is no longer running")

env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")
log_handle = LOG_PATH.open("w")
process = subprocess.Popen(
    [
        "uv",
        "run",
        "nat",
        "mcp",
        "serve",
        "--config_file",
        str(MCP_CONFIG_PATH),
        "--host",
        MCP_HOST,
        "--port",
        str(MCP_PORT),
    ],
    cwd=str(AGENT_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
VSS_ORCHESTRATOR_MCP_PID = process.pid
time.sleep(5)
print(f"Started MCP server with PID {VSS_ORCHESTRATOR_MCP_PID}")
print("In-memory PID:", VSS_ORCHESTRATOR_MCP_PID)
print("Log file:", LOG_PATH)
print("Endpoint:", MCP_URL)


## 8. Deployment options

After [`deploy_nemoclaw_vss.ipynb`](./deploy_nemoclaw_vss.ipynb) finishes its sandbox policy setup, OpenClaw can reach the orchestrator MCP at `http://host.openshell.internal:<MCP_PORT>/mcp` (default `9902`).

### A. Deploy with OpenClaw prompt

Use these prompts in the OpenClaw UI that was configured by [`deploy_nemoclaw_vss.ipynb`](./deploy_nemoclaw_vss.ipynb).

1. Discover the available tools:
   - `Call the MCP server at http://host.openshell.internal:<MCP_PORT>/mcp and list all the available tools`

2. Generate the resolved compose yaml and env:
   - `Generate the compose yaml and env for <profile> profile`

3. Deploy a profile:
   - `Start docker compose for docker compose id <docker-compose-id>`

4. Inspect the running deployment:
   - `Check the compose status for <docker-compose-ops-id>`

5. Tear it down when done:
   - `Tear down docker_compose_id <docker-compose-id>, keep polling the docker compose status until the shutdown finishes.`

### B. Deploy manually

Run the next cells in order. Each step is separated so you can inspect results before continuing. As you run them, the notebook populates variables such as `DOCKER_COMPOSE_ID`, `DOCKER_UP_OPS_ID`, and `DOCKER_DOWN_OPS_ID`. If you changed `MCP_PORT` from the default, substitute that port in the OpenClaw prompt above.


In [ ]:
import json
import os
import subprocess
import sys

if str(VSS_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(VSS_REPO_DIR))

from deployments.scripts.orchestrator_mcp_helper import OrchestratorTool
from deployments.scripts.orchestrator_mcp_helper import build_vss_ui_url
from deployments.scripts.orchestrator_mcp_helper import poll_compose_op
from deployments.scripts.orchestrator_mcp_helper import require_success
from deployments.scripts.orchestrator_mcp_helper import tool_call

MANUAL_STATUS_TAIL_LINES = 200
MANUAL_LOG_TAIL_LINES = 200

print("MCP_URL:", MCP_URL)
print("PROFILE:", PROFILE)
print("HARDWARE_PROFILE:", HARDWARE_PROFILE)
print("EXTRA_ENV_OVERRIDES:", EXTRA_ENV_OVERRIDES)


### B1. Validate profiles and prerequisites

This step confirms the orchestrator is reachable and that the host passes the prerequisite checks before generating any deployment artifacts.


In [ ]:
profiles_result = require_success(
    tool_call(OrchestratorTool.PROFILES, mcp_url=MCP_URL, agent_dir=AGENT_DIR),
    "profiles",
)

prereqs_result = require_success(
    tool_call(OrchestratorTool.PREREQS, mcp_url=MCP_URL, agent_dir=AGENT_DIR),
    "prereqs",
)

print("Profiles and prerequisites checks completed.")


### B2. Generate the compose bundle

This step resolves the selected profile and saves the generated compose ID plus the rendered `.env` and compose content in notebook memory.


In [ ]:
generate_args = {
    "profile": PROFILE,
    "env_overrides": [
        f"HARDWARE_PROFILE={HARDWARE_PROFILE}",
        *EXTRA_ENV_OVERRIDES,
    ],
}
if NGC_CLI_API_KEY.strip():
    generate_args["ngc_cli_api_key"] = NGC_CLI_API_KEY
if NVIDIA_API_KEY.strip():
    generate_args["nvidia_api_key"] = NVIDIA_API_KEY

generate_result = require_success(
    tool_call(OrchestratorTool.DOCKER_GENERATE, mcp_url=MCP_URL, agent_dir=AGENT_DIR, arguments=generate_args),
    "docker_generate",
)
DOCKER_COMPOSE_ID = generate_result["docker_compose_id"]
print("DOCKER_COMPOSE_ID:", DOCKER_COMPOSE_ID)

read_result = require_success(
    tool_call(
        OrchestratorTool.DOCKER_READ,
        mcp_url=MCP_URL,
        agent_dir=AGENT_DIR,
        arguments={"docker_compose_id": DOCKER_COMPOSE_ID},
        show_response=False,
    ),
    "docker_read",
)
GENERATED_ENV_CONTENT = read_result["env_content"]
GENERATED_COMPOSE_YAML_CONTENT = read_result["compose_yaml_content"]
print("Generated env bytes:", len(GENERATED_ENV_CONTENT.encode("utf-8")))
print("Generated compose bytes:", len(GENERATED_COMPOSE_YAML_CONTENT.encode("utf-8")))


### B3. Start the deployment

This step starts `docker compose` for the generated bundle and stores the operation ID used for later polling.


In [ ]:
if not globals().get("DOCKER_COMPOSE_ID"):
    raise RuntimeError("DOCKER_COMPOSE_ID is not set. Run the compose generation step first.")

up_result = require_success(
    tool_call(
        OrchestratorTool.DOCKER_UP,
        mcp_url=MCP_URL,
        agent_dir=AGENT_DIR,
        arguments={"docker_compose_id": DOCKER_COMPOSE_ID},
    ),
    "docker_up",
)
DOCKER_UP_OPS_ID = up_result["docker_compose_ops_id"]
print("DOCKER_UP_OPS_ID:", DOCKER_UP_OPS_ID)


### B4. Wait for the deployment and inspect the result

This step polls the deployment until it finishes, lists the containers, and prints the VSS UI URL for the current notebook settings.


In [ ]:
if not globals().get("DOCKER_UP_OPS_ID"):
    raise RuntimeError("DOCKER_UP_OPS_ID is not set. Run the deployment start step first.")

UP_STATUS_RESULT = poll_compose_op(
    DOCKER_UP_OPS_ID,
    mcp_url=MCP_URL,
    agent_dir=AGENT_DIR,
    tail_lines=MANUAL_STATUS_TAIL_LINES,
)
print("Final deployment status:", UP_STATUS_RESULT["status"])

list_result = require_success(
    tool_call(
        OrchestratorTool.DOCKER_LIST,
        mcp_url=MCP_URL,
        agent_dir=AGENT_DIR,
        arguments={"all_containers": True},
    ),
    "docker_list",
)
CONTAINER_NAMES = list_result.get("container_names", [])
SELECTED_CONTAINER_NAME = CONTAINER_NAMES[0] if CONTAINER_NAMES else None
print("CONTAINER_NAMES:", CONTAINER_NAMES)
print("SELECTED_CONTAINER_NAME:", SELECTED_CONTAINER_NAME or "unavailable")
VSS_UI_URL = build_vss_ui_url()
print("VSS_UI_URL:", VSS_UI_URL or "unavailable")


### B5. Stop the VSS Deployment

In [ ]:
if not globals().get("DOCKER_COMPOSE_ID"):
    raise RuntimeError("DOCKER_COMPOSE_ID is not set. Run the manual deployment cell first.")

down_result = require_success(
    tool_call(
        OrchestratorTool.DOCKER_DOWN,
        mcp_url=MCP_URL,
        agent_dir=AGENT_DIR,
        arguments={"docker_compose_id": DOCKER_COMPOSE_ID},
    ),
    "docker_down",
)
DOCKER_DOWN_OPS_ID = down_result["docker_compose_ops_id"]
print("DOCKER_DOWN_OPS_ID:", DOCKER_DOWN_OPS_ID)

DOWN_STATUS_RESULT = poll_compose_op(DOCKER_DOWN_OPS_ID, mcp_url=MCP_URL, agent_dir=AGENT_DIR, tail_lines=MANUAL_STATUS_TAIL_LINES, sleep_s=5)
print("Final shutdown status:", DOWN_STATUS_RESULT["status"])


## 9. Stop the VSS Orchestrator MCP server

In [ ]:
import os
import signal

pid = globals().get("VSS_ORCHESTRATOR_MCP_PID")
if pid:
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"Sent SIGTERM to MCP server PID {pid}")
    except ProcessLookupError:
        print(f"Process {pid} is no longer running")
    VSS_ORCHESTRATOR_MCP_PID = None
else:
    print("No in-memory MCP server PID found. Nothing to stop.")
